In [21]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
from pathlib import Path
import pandas as pd

data_dir = Path("../../data")
X_train = pd.read_csv(data_dir / "X_train.csv")
X_test = pd.read_csv(data_dir / "X_test.csv")
y_train = pd.read_csv(data_dir / "y_train.csv").values.ravel()
y_test = pd.read_csv(data_dir / "y_test.csv").values.ravel()

print("Loaded:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Loaded:
X_train: (227972, 33)
X_test:  (10852, 33)
y_train: (227972,)
y_test:  (10852,)


In [23]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

X_all = pd.concat(
    [X_train, X_test],
    ignore_index=True,
)

y_all = np.concatenate(
    [y_train, y_test]
)

final_logistic_model = LogisticRegression(
    C=0.1,
    solver="lbfgs",
    l1_ratio=0,
    max_iter=10000,
)

final_logistic_model.fit(X_all, y_all)

c:\Users\523ba\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,10000
,multi_class,'deprecated'


In [24]:
from pathlib import Path
import joblib

artifacts_dir = Path("../../artifacts")
artifacts_dir.mkdir(exist_ok=True)

joblib.dump(
    final_logistic_model,
    artifacts_dir / "logistic_model.joblib",
)

['..\\..\\artifacts\\logistic_model.joblib']

In [25]:
import json

feature_columns = list(X_all.columns)

with open(
    artifacts_dir / "feature_columns.json",
    "w",
) as f:
    json.dump(
        feature_columns,
        f,
        indent=4,
    )

In [28]:
import json
import random
from pathlib import Path

import joblib
import numpy as np

from src.inference.predict import predict_matchup
from src.inference.team_lookup import (
    load_final_team_states,
    get_team_state,
)
from src.inference.matchup_features import build_matchup_features


# ============================================================
# Setup
# ============================================================

PROJECT_ROOT = Path("../..")

MODEL_PATH = PROJECT_ROOT / "artifacts" / "logistic_model.joblib"
FEATURE_COLUMNS_PATH = PROJECT_ROOT / "artifacts" / "feature_columns.json"

states = load_final_team_states()
model = joblib.load(MODEL_PATH)

with open(FEATURE_COLUMNS_PATH) as f:
    feature_columns = json.load(f)

TOL = 1e-10

print("=" * 60)
print("INFERENCE SANITY CHECKS")
print("=" * 60)


# ============================================================
# 1. Artifact checks
# ============================================================

assert MODEL_PATH.exists(), "Model file does not exist."
assert FEATURE_COLUMNS_PATH.exists(), "Feature-columns file does not exist."

assert len(feature_columns) > 0
assert len(feature_columns) == len(set(feature_columns)), (
    "Duplicate feature names found."
)

assert hasattr(model, "predict_proba"), (
    "Loaded model does not support predict_proba."
)

assert model.n_features_in_ == len(feature_columns), (
    f"Model expects {model.n_features_in_} features, "
    f"but feature_columns contains {len(feature_columns)}."
)

print("✓ Artifacts valid")


# ============================================================
# 2. Final-team-state table checks
# ============================================================

required_metadata = {"TeamID", "TeamName", "Season"}

assert required_metadata.issubset(states.columns), (
    "Missing required metadata columns."
)

assert not states.empty
assert states["TeamID"].notna().all()
assert states["TeamName"].notna().all()
assert states["Season"].notna().all()

# Every team-season should appear exactly once.
duplicates = states.duplicated(
    subset=["TeamID", "Season"],
    keep=False,
)

assert not duplicates.any(), (
    "Duplicate TeamID/Season rows found."
)

print("✓ Final team-state table valid")


# ============================================================
# Choose two valid team-seasons for deterministic checks
# ============================================================

team_a_name = "Duke"
team_a_season = 2003

team_b_name = "Duke"
team_b_season = 2004

team_a = get_team_state(
    states,
    team_a_name,
    team_a_season,
)

team_b = get_team_state(
    states,
    team_b_name,
    team_b_season,
)

print("✓ Team lookup works")


# ============================================================
# 3. Full matchup construction
# ============================================================

X_matchup = build_matchup_features(
    team_1_state=team_a,
    team_2_state=team_b,
    team_1_location=0,
)

assert len(X_matchup) == 1
assert X_matchup.columns.is_unique

missing_features = [
    col for col in feature_columns
    if col not in X_matchup.columns
]

assert not missing_features, (
    f"Inference matchup is missing model features: "
    f"{missing_features}"
)

X_selected = X_matchup[feature_columns]

assert list(X_selected.columns) == feature_columns
assert X_selected.shape[1] == model.n_features_in_

print("✓ Matchup feature construction valid")
print(f"  Full matchup features: {X_matchup.shape[1]}")
print(f"  Selected model features: {X_selected.shape[1]}")


# ============================================================
# 4. No NaN / infinity in selected model input
# ============================================================

values = X_selected.to_numpy(dtype=float)

assert np.isfinite(values).all(), (
    "NaN or infinity found in inference features."
)

print("✓ Model inputs contain no NaN/inf")


# ============================================================
# 5. Direct model prediction validity
# ============================================================

proba = model.predict_proba(X_selected)

assert proba.shape == (1, 2)
assert np.all(proba >= 0)
assert np.all(proba <= 1)
assert abs(proba.sum() - 1.0) < TOL

print("✓ Direct predict_proba valid")


# ============================================================
# 6. Public predict_matchup() output
# ============================================================

result = predict_matchup(
    team_1_name=team_a_name,
    team_1_season=team_a_season,
    team_2_name=team_b_name,
    team_2_season=team_b_season,
    team_1_location=0,
)

p1 = result["team_1_win_probability"]
p2 = result["team_2_win_probability"]

assert 0 <= p1 <= 1
assert 0 <= p2 <= 1
assert abs(p1 + p2 - 1.0) < TOL

# Public function should agree with direct model call.
assert abs(p1 - proba[0, 1]) < TOL
assert abs(p2 - proba[0, 0]) < TOL

print("✓ predict_matchup output valid")


# ============================================================
# 7. Neutral-team swap symmetry
# ============================================================

forward = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    0,
)

reverse = predict_matchup(
    team_b_name,
    team_b_season,
    team_a_name,
    team_a_season,
    0,
)

assert abs(
    forward["team_1_win_probability"]
    - reverse["team_2_win_probability"]
) < TOL

assert abs(
    forward["team_2_win_probability"]
    - reverse["team_1_win_probability"]
) < TOL

print("✓ Neutral swap symmetry valid")


# ============================================================
# 8. Home/away swap symmetry
# ============================================================

forward_home = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    1,
)

reverse_away = predict_matchup(
    team_b_name,
    team_b_season,
    team_a_name,
    team_a_season,
    -1,
)

assert abs(
    forward_home["team_1_win_probability"]
    - reverse_away["team_2_win_probability"]
) < TOL

assert abs(
    forward_home["team_2_win_probability"]
    - reverse_away["team_1_win_probability"]
) < TOL

print("✓ Home/away swap symmetry valid")


# ============================================================
# 9. Same team-season on neutral court
# ============================================================

same_team = predict_matchup(
    team_a_name,
    team_a_season,
    team_a_name,
    team_a_season,
    0,
)

same_p = same_team["team_1_win_probability"]

print(
    f"  Same-team neutral probability: "
    f"{same_p:.6f}"
)

# With mirrored training this should be essentially 0.5.
assert abs(same_p - 0.5) < 1e-6, (
    "Same team-season on neutral court is not ~50/50."
)

print("✓ Same-team neutral check valid")


# ============================================================
# 10. Location should actually affect predictions
# ============================================================

neutral = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    0,
)["team_1_win_probability"]

home = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    1,
)["team_1_win_probability"]

away = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    -1,
)["team_1_win_probability"]

print(
    f"  Away={away:.4f}, "
    f"Neutral={neutral:.4f}, "
    f"Home={home:.4f}"
)

assert home > neutral > away, (
    "Expected home > neutral > away probability."
)

print("✓ Location behaves sensibly")


# ============================================================
# 11. Invalid location rejected
# ============================================================

try:
    predict_matchup(
        team_a_name,
        team_a_season,
        team_b_name,
        team_b_season,
        7,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Invalid team_1_location was not rejected."
    )

print("✓ Invalid location rejected")


# ============================================================
# 12. Invalid team-season rejected
# ============================================================

try:
    predict_matchup(
        team_1_name="Definitely Not A Team",
        team_1_season=9999,
        team_2_name=team_b_name,
        team_2_season=team_b_season,
        team_1_location=0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Invalid team-season was not rejected."
    )

print("✓ Invalid team-season rejected")


# ============================================================
# 13. Random batch smoke test
# ============================================================

records = states[
    ["TeamName", "Season"]
].drop_duplicates().to_records(index=False)

random.seed(42)

for _ in range(100):
    a, b = random.sample(list(records), 2)

    r = predict_matchup(
        team_1_name=str(a[0]),
        team_1_season=int(a[1]),
        team_2_name=str(b[0]),
        team_2_season=int(b[1]),
        team_1_location=random.choice([-1, 0, 1]),
    )

    p1 = r["team_1_win_probability"]
    p2 = r["team_2_win_probability"]

    assert np.isfinite(p1)
    assert np.isfinite(p2)
    assert 0 <= p1 <= 1
    assert 0 <= p2 <= 1
    assert abs(p1 + p2 - 1.0) < TOL

print("✓ 100 random matchup predictions passed")


# ============================================================
# Done
# ============================================================

print()
print("=" * 60)
print("ALL INFERENCE SANITY CHECKS PASSED")
print("=" * 60)

INFERENCE SANITY CHECKS
✓ Artifacts valid
✓ Final team-state table valid
✓ Team lookup works
✓ Matchup feature construction valid
  Full matchup features: 63
  Selected model features: 33
✓ Model inputs contain no NaN/inf
✓ Direct predict_proba valid
✓ predict_matchup output valid
✓ Neutral swap symmetry valid
✓ Home/away swap symmetry valid
  Same-team neutral probability: 0.500000
✓ Same-team neutral check valid
  Away=0.3143, Neutral=0.4599, Home=0.6126
✓ Location behaves sensibly
✓ Invalid location rejected
✓ Invalid team-season rejected
✓ 100 random matchup predictions passed

ALL INFERENCE SANITY CHECKS PASSED


Project: NCAA Historical Matchup Predictor Web App

Goal:
Build a simple web app where a user selects:
- Team 1
- Team 1 season
- Team 2
- Team 2 season
- Team 1 location: Home / Neutral / Away

The app should call the existing inference backend and display both teams' model-estimated win probabilities.

Important: the ML/backend inference is already finished. Do not retrain models or recreate feature engineering in the web app.

Backend assets already available:

1. data/final_team_states.csv
   - One row per team-season.
   - Contains:
     TeamID
     TeamName
     Season
     full end-of-regular-season feature state
   - This file should also be used to populate valid team/season dropdowns.

2. artifacts/logistic_model.joblib
   - Fully fitted final LogisticRegression model.
   - Trained on all available labeled data after model evaluation was completed.

3. artifacts/feature_columns.json
   - Exact selected feature names, in exact order, expected by the saved logistic regression.
   - Current deployed model uses 35 selected features.

Existing inference package:

src/inference/
    team_lookup.py
    matchup_features.py
    predict.py

Public backend function:

predict_matchup(
    team_1_name: str,
    team_1_season: int,
    team_2_name: str,
    team_2_season: int,
    team_1_location: int = 0,
) -> dict

Location coding:
    1  = Team 1 home
    0  = neutral
   -1  = Team 1 away

Example:

result = predict_matchup(
    team_1_name="Duke",
    team_1_season=2003,
    team_2_name="Duke",
    team_2_season=2004,
    team_1_location=0,
)

Expected result structure:

{
    "team_1": "Duke 2003",
    "team_2": "Duke 2004",
    "team_1_win_probability": 0.4477,
    "team_2_win_probability": 0.5523
}

The inference flow already works as follows:

saved team-season states
    ↓
lookup Team 1 + Team 2
    ↓
construct full differential matchup row
    ↓
select exact saved feature_columns
    ↓
saved logistic regression
    ↓
predict_proba()
    ↓
return both probabilities

This inference layer has already passed sanity tests:
- probabilities sum to 1
- neutral team-order reversal gives complementary probabilities
- home/away reversal gives complementary probabilities
- identical team-season vs itself on neutral gives exactly ~50/50
- location affects predictions sensibly
- invalid team-season combinations are rejected
- invalid locations are rejected
- 100 random matchup predictions completed successfully

Web-app requirements:

Use final_team_states.csv as the source of truth for valid selections.

The UI should prevent invalid team-season combinations. For example:
- after choosing a team, only show seasons that exist for that team
OR
- after choosing a season, only show teams that exist in that season

Inputs:
- Team 1 dropdown
- Team 1 season dropdown
- Team 2 dropdown
- Team 2 season dropdown
- Location selector:
    Team 1 Home
    Neutral
    Team 1 Away
- Predict button

On click:
- call predict_matchup(...)
- display both probabilities prominently
- optionally highlight the predicted winner

Example display:

Duke 2003        63.2%
Duke 2004        36.8%

Predicted winner: Duke 2003

Important architectural constraint:
The web app should be a thin presentation layer. It should import and call src.inference.predict.predict_matchup rather than duplicating:
- team lookup
- differential feature construction
- feature filtering
- model loading logic
- ML training

The deployed app only needs:
- app code
- src/inference/
- data/final_team_states.csv
- artifacts/logistic_model.joblib
- artifacts/feature_columns.json
- Python dependencies

Training notebooks, raw NCAA source files, X_train/X_test, feature-selection code, and other experimental models are not required at runtime.

Recommended MVP:
Streamlit is perfectly suitable because this is primarily dropdown selection + prediction output.

Main objective:
Create a clean, polished historical "what-if" NCAA matchup experience, e.g.

Duke 2019 vs Duke 2024
UConn 2023 vs Kentucky 2012
etc.

The displayed percentages should be described as "model-estimated win probability," since these are hypothetical cross-season matchups.